# Semantic Segmentation with a U-Net (Oxford-IIIT Pet)

A first **segmentation** study: instead of one label per image, predict a class for every pixel. We use the Oxford-IIIT Pet trimaps (foreground, background, boundary) and a small U-Net encoder-decoder. This is a scaffold to build on: the data pipeline, a baseline model, and a short training loop are here; tuning, augmentation, and a proper Dice/IoU report are the next steps.

> The download is large (~800 MB) and training wants a GPU, so run this on Colab.

## Setup

In [ ]:
# One-time setup: make the `visualization` helper importable, then fetch data +
# resolve paths. Each study's fetch logic lives in its own download_data.py.
import os
import sys

if "google.colab" in sys.modules:
    if not os.path.isdir("ConvolutedComputerVision"):
        !git clone -q https://github.com/samlowe106/ConvolutedComputerVision.git
    sys.path.insert(0, "ConvolutedComputerVision/src")

from visualization import colab_bootstrap

DATA_ROOT, CKPT_ROOT = colab_bootstrap(study="oxford-pet-segmentation")

In [ ]:
import datetime
import glob

import matplotlib.pyplot as plt
import tensorflow as tf

notebook_start_time = datetime.datetime.now()

## Data: images and trimap masks

In [ ]:
IMG_SIZE = 128
image_dir = os.path.join(DATA_ROOT, "images")
mask_dir = os.path.join(DATA_ROOT, "annotations", "trimaps")

stems = sorted(
    os.path.splitext(os.path.basename(p))[0]
    for p in glob.glob(os.path.join(image_dir, "*.jpg"))
)
image_paths = [os.path.join(image_dir, s + ".jpg") for s in stems]
mask_paths = [os.path.join(mask_dir, s + ".png") for s in stems]


def _decodable(path):
    # a handful of Oxford-IIIT Pet files are not valid JPEGs; a try/except lets us skip them
    try:
        tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
        return True
    except tf.errors.InvalidArgumentError:
        return False


pairs = [(ip, mp) for ip, mp in zip(image_paths, mask_paths) if _decodable(ip)]
image_paths = [ip for ip, _ in pairs]
mask_paths = [mp for _, mp in pairs]
print(f"{len(image_paths)} decodable images ({len(stems) - len(image_paths)} skipped)")


def _load(image_path, mask_path):
    img = tf.io.decode_jpeg(tf.io.read_file(image_path), channels=3)
    img = tf.cast(tf.image.resize(img, (IMG_SIZE, IMG_SIZE)), tf.uint8)
    mask = tf.io.decode_png(tf.io.read_file(mask_path), channels=1)
    mask = tf.image.resize(mask, (IMG_SIZE, IMG_SIZE), method="nearest")
    mask = tf.cast(mask, tf.int32) - 1  # trimap {1,2,3} -> classes {0,1,2}
    return img, mask


full = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths)).map(
    _load, num_parallel_calls=tf.data.AUTOTUNE
)
n_val = 500
val_ds = full.take(n_val).batch(32).prefetch(tf.data.AUTOTUNE)
train_ds = full.skip(n_val).shuffle(1000).batch(32).prefetch(tf.data.AUTOTUNE)

In [ ]:
# preview one image and its mask
img, mask = next(iter(full.take(1)))
fig, (a0, a1) = plt.subplots(1, 2, figsize=(8, 4))
a0.imshow(img.numpy())
a0.set_title("image")
a0.axis("off")
a1.imshow(mask.numpy().squeeze())
a1.set_title("trimap mask")
a1.axis("off")
plt.show()

## Model: a small U-Net

In [ ]:
def build_unet(size=IMG_SIZE, n_classes=3):
    inputs = tf.keras.Input((size, size, 3))
    x = tf.keras.layers.Rescaling(1.0 / 255)(inputs)
    skips = []
    for filters in (32, 64, 128):
        x = tf.keras.layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = tf.keras.layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        skips.append(x)
        x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.Conv2D(256, 3, padding="same", activation="relu")(x)
    for filters, skip in zip((128, 64, 32), reversed(skips)):
        x = tf.keras.layers.Conv2DTranspose(filters, 2, strides=2, padding="same")(x)
        x = tf.keras.layers.Concatenate()([x, skip])
        x = tf.keras.layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = tf.keras.layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    outputs = tf.keras.layers.Conv2D(n_classes, 1, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name="unet")


model = build_unet()
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)
model.summary()

## Train (baseline) and next steps

A short run to confirm the pipeline learns. To turn this into a real study: add augmentation, track **mean IoU** and a **Dice** loss (pixel accuracy is misleading when background dominates), save the best model to `CKPT_ROOT`, and overlay predicted masks on a few validation images.

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=5)
# TODO: mean IoU / Dice, augmentation, checkpointing, prediction overlays

In [ ]:
notebook_end_time = datetime.datetime.now()
print(
    f"Notebook last run (end-to-end): {notebook_end_time} "
    f"(duration: {notebook_end_time - notebook_start_time})"
)